# Análisis del índice de calor en Iquitos (2000–2024)

Este notebook parte del dataset limpio generado en `01_exploracion.ipynb`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/iquitos_horario.csv", parse_dates=["fecha_hora"])
df["anio"] = df["fecha_hora"].dt.year
df["mes"] = df["fecha_hora"].dt.month
df["hora"] = df["fecha_hora"].dt.hour
df.head()

,fecha_hora,temperatura,punto_rocio,anio,mes,hora
0,2000-01-01 00:00:00-05:00,25.0,24.0,2000,1,0
1,2000-01-01 01:00:00-05:00,25.0,23.6,2000,1,1
2,2000-01-01 02:00:00-05:00,25.0,24.0,2000,1,2
3,2000-01-01 03:00:00-05:00,24.0,23.0,2000,1,3
4,2000-01-01 04:00:00-05:00,23.0,23.0,2000,1,4


$$HR = 100 \times \frac{e^{\frac{17.625 \cdot T_d}{243.04 + T_d}}}{e^{\frac{17.625 \cdot T}{243.04 + T}}}$$

In [2]:
# Fórmula de Magnus: humedad relativa a partir de la temperatura y el punto de rocío
numerador = np.exp((17.625 * df["punto_rocio"]) / (243.04 + df["punto_rocio"]))

denominador = np.exp((17.625 * df["temperatura"]) / (243.04 + df["temperatura"]))

df["humedad_relativa"] = 100 * (numerador / denominador)


6867  horas saturadas


In [3]:
df[df["humedad_relativa"] < 0]

,fecha_hora,temperatura,punto_rocio,anio,mes,hora,humedad_relativa


In [4]:
df[df["punto_rocio"] == df["temperatura"]]

,fecha_hora,temperatura,punto_rocio,anio,mes,hora,humedad_relativa
4,2000-01-01 04:00:00-05:00,23.0,23.0,2000,1,4,100.0
5,2000-01-01 05:00:00-05:00,23.0,23.0,2000,1,5,100.0
6,2000-01-01 06:00:00-05:00,23.0,23.0,2000,1,6,100.0
28,2000-01-02 04:00:00-05:00,23.0,23.0,2000,1,4,100.0
29,2000-01-02 05:00:00-05:00,23.0,23.0,2000,1,5,100.0
...,...,...,...,...,...,...,...
197381,2024-12-28 06:00:00-05:00,23.0,23.0,2024,12,6,100.0
197405,2024-12-29 06:00:00-05:00,24.0,24.0,2024,12,6,100.0
197428,2024-12-30 05:00:00-05:00,23.0,23.0,2024,12,5,100.0
197429,2024-12-30 06:00:00-05:00,23.0,23.0,2024,12,6,100.0


Una humedad relativa mayor a 100% es imposible, ya que el aire no puede contener más vapor del que lo satura. Se tiene que verificar

In [5]:
mayores = df[df["humedad_relativa"] > 100]
print(len(mayores), "horas con HR > 100%")
print(round(len(mayores) / len(df) * 100, 2), "% del total")

0 horas con HR > 100%
0.0 % del total


In [6]:
(mayores["punto_rocio"] - mayores["temperatura"]).describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
dtype: float64

In [9]:
df["humedad_relativa"].describe()

count    197471.000000
mean         84.591474
std          12.088424
min          30.836886
25%          77.715244
50%          88.679561
75%          94.107970
max         100.000000
Name: humedad_relativa, dtype: float64

In [10]:
print((df["humedad_relativa"] == 100).sum(), " horas saturadas")

6867  horas saturadas


La humedad relativa mediana es de 88.7%; el 75% supera el 77.7%.

## Índice de calor

Se usa el algoritmo del Servicio Meteorológico Nacional de EE. UU. (NWS),
que combina temperatura y humedad para estimar la sensación térmica.
El algoritmo trabaja en grados Fahrenheit y usa dos fórmulas:

- **Fórmula simple:** para condiciones templadas.
- **Regresión de Rothfusz:** para calor intenso (cuando el promedio entre
  la fórmula simple y la temperatura es de 80 °F o más).

Se aplica el ajuste del NWS para humedad mayor a 85%. El ajuste para
humedad menor a 13% se omite porque no ocurre en Iquitos.

In [11]:
T = df["temperatura"] * 9/5 + 32
HR = df["humedad_relativa"]

In [12]:
ic_simple = 0.5 * (T + 61.0 + (T - 68.0) * 1.2 + HR * 0.094)

In [13]:
ic_rothfusz = (-42.379 + 2.04901523*T + 10.14333127*HR
            - 0.22475541*T*HR - 0.00683783*T**2
            - 0.05481717*HR**2 + 0.00122874*T**2*HR
            + 0.00085282*T*HR**2 - 0.00000199*T**2*HR**2)

ajuste = ((HR - 85) / 10) * ((87 - T) / 5)
condicion_ajuste = (HR > 85) & (T >= 80) & (T <= 87)
ic_rothfusz = np.where(condicion_ajuste, ic_rothfusz + ajuste, ic_rothfusz)

In [14]:
ic_rothfusz

array([77.96211196, 78.08357739, 77.96211196, ..., 77.96211196,
       77.99361546, 77.96211196], shape=(197471,))

Convertir a Celsius